In [1]:
from neo4j import GraphDatabase, RoutingControl

URI = "neo4j://127.0.0.1:7687"
AUTH = ("neo4j", "Shellatiwi123")

In [2]:
def add_friend(driver, name, friend_name):
    driver.execute_query(
        "MERGE (a:Person {name: $name}) "
        "MERGE (friend:Person {name: $friend_name}) "
        "MERGE (a)-[:KNOWS]->(friend)",
        name=name, friend_name=friend_name, database_="neo4j"
    )

In [3]:
def print_friends(driver, name):
    records, _, _ = driver.execute_query(
        "MATCH (a:Person)-[:KNOWS]->(friend) WHERE a.name = $name "
        "RETURN friend.name ORDER BY friend.name",
        name=name, database_="neo4j", routing_=RoutingControl.READ,
    )
    for record in records:
        print(record["friend.name"])

with GraphDatabase.driver(URI, auth=AUTH, ) as driver:
    add_friend(driver, "Arthur", "Guinevere")
    add_friend(driver, "Arthur", "Lancelot")
    add_friend(driver, "Arthur", "Merlin")
    print_friends(driver, "Arthur")

Guinevere
Lancelot
Merlin


# Pertanyaan Pendahuluan 
### 1. Apa nama database yang digunakan pada tutorial ini? 

Neo4j Terlihat dari parameter 'database_="neo4j"` pada
`execute_query`. Ini database default dalam sebuah instance Neo4j. Nama instance di Desktop (`UTSGSD`) adalah nama DBMS, bukan nama database.

### 2. Sebutkan akses lain (seperti sintaks DDL/DML CYPHER) yang dapat digunakan mengakses database lain dalam 1 instance!

a. Driver (parameter `execute_query`):
   `driver.execute_query(query, database_="namadb")`

b. Driver (session):
   `driver.session(database="namadb")`
   (di bagian 1.3 dipakai `database="system"`)

c. Cypher (Query tool / cypher-shell):
   `USE namadb MATCH (n) RETURN n;`
   atau perintah browser `:use namadb`

d. Administrasi (dijalankan pada database `system`):
   `SHOW DATABASES;`
   `CREATE DATABASE namadb;`
   `DROP DATABASE namadb;`

### 3. Apa jenis graf pertemanan yang telah dibuat di sini? 

Graf berarah (directed graph).Karena, relasi yang dipakai adalah `-[:KNOWS]->`, jadi ada arah panahnya, dari Arthur ke temannya. Artinya yang tersimpan adalah "Arthur mengenal Guinevere", bukan sebaliknya.

Kalau dilihat dari datanya, ada 4 node berlabel `Person` (Arthur, Guinevere, Lancelot, Merlin) dan 3 relasi `KNOWS`. Semua node punya label yang sama dan semua relasi bertipe sama, jadi grafnya **homogen**. Relasinya juga tidak punya properti seperti bobot, jadi grafnya **tidak berbobot**.

Bentuknya seperti **bintang (star)**: Arthur di tengah dengan 3 panah keluar (out-degree 3), sedangkan tiga temannya masing-masing hanya menerima 1 panah (in-degree 1) dan tidak punya panah keluar. Tidak ada siklus.